# Custom Compresr Tools — VC Analyst Research Agent

You are a venture analyst at an early-stage fund. You’re writing an investment memo on **Anthropic** for the partnership next week and you need fast, grounded answers from primary sources.

Three sections, one workflow:

1. **Web search tool (Tavily)** — live search for recent news, results flow through Compresr.
2. **Page-fetch tool (Anthropic LLM)** — pull a long primary source, extract memo-ready facts. Off vs on **`toc_latte_v1`** (tool-output compression) side by side.
3. **Same comparison with OpenAI** — provider swap is one line.

You bring three keys: `COMPRESR_API_KEY`, the LLM provider key, and a search key (Tavily or Brave).

## Setup


In [1]:
# pip install "compresr[agents-all]"
import os
from pathlib import Path
from dotenv import load_dotenv

for parent in [Path.cwd(), *Path.cwd().parents]:
    if (parent / ".env").exists():
        load_dotenv(parent / ".env")
        break

from compresr import CompressionClient, WebSearchTool
from langchain_core.tools import tool


## Construct the client

Pass just the **provider** — the model lives at the call site, matching `anthropic.Anthropic()` and `openai.OpenAI()`. (Use `llm="anthropic:claude-haiku-4-5"` to bake in a default; the call-site `model=` always wins.)


In [2]:
client = CompressionClient(
    api_key=os.environ["COMPRESR_API_KEY"],
    llm="anthropic",
    llm_api_key=os.environ["ANTHROPIC_API_KEY"],
    compression={
        "compression_model_name": "latte_v1",
        "target_compression_ratio": 0.5,
        "min_tokens": 500,
    },
)

## 1. Web search tool — Tavily

Every web search dumps thousands of tokens of search snippets into the LLM context. `toc_latte_v1` compresses each tool output against the user’s question *before* the LLM re-reads it.

`WebSearchTool` returns a real LangChain `BaseTool`. Two interchangeable construction forms:

| Form | Example |
|---|---|
| Factory classmethod | `WebSearchTool.tavily(api_key=..., max_results=3)` |
| Provider arg | `WebSearchTool(provider="tavily", api_key=..., max_results=3)` |

Same shape for Brave: `WebSearchTool.brave(...)` ≡ `WebSearchTool(provider="brave", ...)`.

In [3]:
tavily = WebSearchTool.tavily(
    api_key=os.environ["TAVILY_API_KEY"],
    max_results=5,
)

ANALYST_SEARCH_QUERY = (
    "I am a VC analyst writing an investment memo on Anthropic. "
    "Use tavily_search to find news from the last 12 months on Anthropic’s "
    "latest funding round, valuation, and lead investors. "
    "Give me a 3-bullet summary with dollar amounts and dates I can paste into a memo."
)

response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=400,
    messages=[{"role": "user", "content": ANALYST_SEARCH_QUERY}],
    tools=[tavily],
)
print(response.content[0].text)

/Users/oussama/anaconda3/lib/python3.11/site-packages/langgraph/checkpoint/base/__init__.py:17: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


Here is your memo-ready 3-bullet summary, sourced from Bloomberg, TechCrunch, Anthropic's official announcement, and The SaaS News:

---

## Anthropic – Funding & Valuation Summary *(for Investment Memo)*

- **Series E | March 3, 2025 | $3.5B raised | $61.5B post-money valuation** — Led by **Lightspeed Venture Partners**, with co-investors including Bessemer Venture Partners, Cisco Investments, D1 Capital Partners, Fidelity Management & Research, General Catalyst, Jane Street, Menlo Ventures, Salesforce Ventures, and Google. Proceeds earmarked for next-gen AI development, compute capacity expansion, safety/alignment research, and international growth.

- **Series F | September 2, 2025 | $13B raised | $183B post-money valuation** — Led by **ICONIQ Capital**, co-led by **Fidelity Management & Research** and **Lightspeed Venture Partners**. Broad syndicate included Altimeter, Baillie Gifford, BlackRock, Blackstone, Coatue, D1 Capital, General Atlantic, General Catalyst, GIC (Singapore), G

In [4]:
print("Brave swap-in (uncomment when you have credits):")
print()
print("    brave = WebSearchTool.brave(api_key=os.environ[\"BRAVE_API_KEY\"], max_results=5)")
print("    response = client.messages.create(")
print("        model=\"claude-haiku-4-5\",")
print("        max_tokens=400,")
print("        messages=[{\"role\": \"user\", \"content\": ANALYST_SEARCH_QUERY}],")
print("        tools=[brave],")
print("    )")

Brave swap-in (uncomment when you have credits):

    brave = WebSearchTool.brave(api_key=os.environ["BRAVE_API_KEY"], max_results=5)
    response = client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=400,
        messages=[{"role": "user", "content": ANALYST_SEARCH_QUERY}],
        tools=[brave],
    )


## 2. Page-fetch tool — Anthropic, off vs on (full benchmark)

The analyst's next step: pull a long primary-source bundle (a stitched corpus of Anthropic + Claude + Dario Amodei + Constitutional AI Wikipedia pages — ~120 KB / ~30k tokens) and extract memo-ready facts. This is the workload where **`toc_latte_v1`** earns its keep — one tool call returns a single large chunk that exceeds `min_tokens`, so the middleware fires on the whole bundle and only the parts relevant to the memo question survive into the model context.

After the benchmark we also display a **GitHub-style word-level diff** showing exactly what the compression dropped vs kept.

In [5]:
import time
import sys
sys.path.insert(0, '.')
from _demo_utils import fetch_wikipedia, compresr_diff_html
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent

RESEARCH_TITLES = ['Anthropic', 'Claude (language model)', 'Dario Amodei', 'Constitutional AI']

@tool
def research_corpus(topic: str) -> str:
    """Fetch a primary-source research corpus on Anthropic. Returns a long
    stitched bundle of related Wikipedia articles (~30k tokens). Use this when
    asked to research a company for an investment memo."""
    parts = []
    for title in RESEARCH_TITLES:
        text = fetch_wikipedia(title)
        if text:
            parts.append(f'# {title}\n\n{text}')
    return '\n\n'.join(parts)

PROMPT = (
    'You are a venture analyst writing an investment memo on Anthropic. '
    'Use the research_corpus tool (pass topic="Anthropic") to fetch primary-source material. '
    'Based ONLY on the fetched corpus, produce a memo-ready briefing with three sections: '
    '(1) Funding history — every round you can find with date, amount, and lead investors; '
    '(2) Key people — founders and notable hires with their prior roles; '
    '(3) Product launches — Claude model releases with dates. '
    'Use only what research_corpus returns; do not search the web separately.'
)

PRICING = {
    'claude-sonnet-4-6': {'input': 3.00, 'output': 15.00},
    'gpt-4o-mini':      {'input': 0.15, 'output': 0.60},
}

def usage_cost(model, usage):
    p = PRICING[model]
    return (usage.get('input_tokens', 0) * p['input'] +
            usage.get('output_tokens', 0) * p['output']) / 1_000_000

_capture = {'pairs': []}
def _install_spy(c):
    orig = c.compress
    def spy(**kw):
        r = orig(**kw)
        _capture['pairs'].append((kw.get('context', ''), r.data.compressed_context))
        return r
    c.compress = spy

_install_spy(client)

def run_bare(provider, model, llm_key):
    chat = init_chat_model(f'{provider}:{model}', api_key=llm_key)
    agent = create_agent(model=chat, tools=[research_corpus])
    t0 = time.perf_counter()
    state = agent.invoke({'messages': [{'role': 'user', 'content': PROMPT}]})
    latency = time.perf_counter() - t0
    ai = state['messages'][-1]
    text = ai.content if isinstance(ai.content, str) else str(ai.content)
    usage = getattr(ai, 'usage_metadata', None) or {}
    return text, usage, latency

def run_compresr(comp_client, model):
    t0 = time.perf_counter()
    r = comp_client.run(prompt=PROMPT, model=model, tools=[research_corpus], max_tokens=600)
    latency = time.perf_counter() - t0
    return r.text, r.usage or {}, latency

JUDGE_MODEL = init_chat_model('anthropic:claude-sonnet-4-6',
                              api_key=os.environ['ANTHROPIC_API_KEY'])

def judge(answer_a, answer_b):
    prompt = (
        f'You are evaluating two answers to the SAME complex query.\n\n'
        f'QUERY:\n{PROMPT}\n\n'
        f'---\nANSWER A (Compresr OFF):\n{answer_a}\n\n'
        f'---\nANSWER B (Compresr ON):\n{answer_b}\n\n'
        f'---\nScore each answer 1-10 on factual accuracy, completeness, and memo-readiness.\n'
        f'Output exactly:\n'
        f'A: <score>/10 — <one-line rationale>\n'
        f'B: <score>/10 — <one-line rationale>\n'
        f'Verdict: <A wins | B wins | Tie> — <one-line rationale>\n'
    )
    return JUDGE_MODEL.invoke(prompt).content

def report(label, model, bare, comp):
    bare_text, bare_usage, bare_lat = bare
    comp_text, comp_usage, comp_lat = comp
    bare_cost = usage_cost(model, bare_usage)
    comp_cost = usage_cost(model, comp_usage)
    h_off = 'without Compresr'
    h_on = 'with Compresr'

    def row(k, a, b):
        print(f'{k:<22} {a:>16}    {b:>16}')

    print(f'{label:<22} {h_off:>16}    {h_on:>16}')
    print('-' * 60)
    row('latency (s)', f'{bare_lat:.2f}', f'{comp_lat:.2f}')
    bare_in = bare_usage.get('input_tokens', 0)
    comp_in = comp_usage.get('input_tokens', 0)
    row('input_tokens', bare_in, comp_in)
    row('output_tokens', bare_usage.get('output_tokens', 0), comp_usage.get('output_tokens', 0))
    D = chr(36)
    row('cost (USD)', f'{D}{bare_cost:.5f}', f'{D}{comp_cost:.5f}')
    saved = bare_cost - comp_cost
    pct = 100 * saved / max(bare_cost, 1e-9)
    input_pct = 100 * (bare_in - comp_in) / max(bare_in, 1)
    NL = chr(10)
    print(NL + f'Input tokens saved: {bare_in - comp_in} ({input_pct:.1f}%)')
    print(f'Cost saved: {D}{saved:.5f} ({pct:.1f}%)')
    print(f'Projected savings per 1,000 requests @ list price: {D}{saved * 1000:.2f}')
    print(NL + '--- Without Compresr (first 320 chars) ---' + NL + bare_text[:320])
    print(NL + '--- With Compresr (first 320 chars) ---' + NL + comp_text[:320])
    print(NL + '--- LLM-as-judge ---' + NL + judge(bare_text, comp_text))

In [6]:
anthropic_bare = run_bare("anthropic", "claude-sonnet-4-6", os.environ["ANTHROPIC_API_KEY"])
anthropic_comp = run_compresr(client, "claude-sonnet-4-6")
report("Anthropic Sonnet 4.6", "claude-sonnet-4-6", anthropic_bare, anthropic_comp)

Anthropic Sonnet 4.6   without Compresr       with Compresr
------------------------------------------------------------
latency (s)                       50.64               18.04
input_tokens                      15382                 719
output_tokens                      2310                 656
cost (USD)                     $0.08080            $0.01200

Input tokens saved: 14663 (95.3%)
Cost saved: $0.06880 (85.2%)
Projected savings per 1,000 requests @ list price: $68.80

--- Without Compresr (first 320 chars) ---
The corpus has been retrieved. Here is the full memo-ready briefing, drawn exclusively from that material.

---

# INVESTMENT MEMO BRIEFING — ANTHROPIC PBC
*All information sourced exclusively from the research corpus returned above.*

---

## SECTION 1 — FUNDING HISTORY

| Date | Round / Event | Amount | Lead / Key In

--- With Compresr (first 320 chars) ---
---

# INVESTMENT MEMO BRIEFING: ANTHROPIC PBC
*Based exclusively on corpus fetched via research_corpus tool*




--- LLM-as-judge ---
A: 9/10 — Comprehensive, well-structured memo with all three sections fully populated, detailed tables, and useful supplementary notes; minor concern is speculative future dates but consistent with corpus-only framing.

B: 6/10 — Sections 2 (Key People) and 3 (Product Launches) are entirely missing or incomplete, and the response appears truncated mid-sentence, making it unusable as a standalone memo-ready briefing.

Verdict: A wins — Answer A delivers all three required sections in polished, memo-ready format while Answer B omits two of the three requested sections entirely.


In [7]:
from IPython.display import display
if _capture['pairs']:
    raw_tool, cmp_tool = _capture['pairs'][0]
    print(f'Spy captured {len(_capture["pairs"])} compress() call(s) during the Anthropic benchmark.')
    print(f'First tool output: {len(raw_tool):,} chars raw -> {len(cmp_tool):,} chars compressed')
    print()
    print('Word-level diff of what toc_latte_v1 dropped from the tool output:')
    display(compresr_diff_html(raw_tool, cmp_tool))
else:
    print('No compressions were captured — the tool output may have been below min_tokens.')


Spy captured 1 compress() call(s) during the Anthropic benchmark.
First tool output: 61,913 chars raw -> 31,484 chars compressed

Word-level diff of what toc_latte_v1 dropped from the tool output:


## 3. Same benchmark — OpenAI provider

The analyst doesn't care which LLM runs the memo — only that the answer is grounded and cheap. Provider switch is one constructor argument; the `latte_v1` policy carries over unchanged.

In [8]:
openai_client = CompressionClient(
    api_key=os.environ["COMPRESR_API_KEY"],
    llm="openai",
    llm_api_key=os.environ["OPENAI_API_KEY"],
    compression={
        "compression_model_name": "latte_v1",
        "target_compression_ratio": 0.5,
        "min_tokens": 500,
    },
)
_install_spy(openai_client)
openai_bare = run_bare("openai", "gpt-4o-mini", os.environ["OPENAI_API_KEY"])
openai_comp = run_compresr(openai_client, "gpt-4o-mini")
report("OpenAI gpt-4o-mini", "gpt-4o-mini", openai_bare, openai_comp)

OpenAI gpt-4o-mini     without Compresr       with Compresr
------------------------------------------------------------
latency (s)                       13.86               14.28
input_tokens                      13312                7040
output_tokens                       793                 576
cost (USD)                     $0.00247            $0.00140

Input tokens saved: 6272 (47.1%)
Cost saved: $0.00107 (43.3%)
Projected savings per 1,000 requests @ list price: $1.07

--- Without Compresr (first 320 chars) ---
# Investment Memo: Anthropic

## 1. Funding History

- **May 2021**: Raised **$124 million** in an initial funding round.
- **April 2022**: Received **$580 million**, led by a **$500 million** investment from FTX.
- **October 2023**: Google invested **$500 million**, with a commitment for an additional **$1.5 billion**

--- With Compresr (first 320 chars) ---
# Investment Memo: Anthropic

### 1. Funding History
- **May 2021**: Raised **$124 million**.
- **April 2022**: A


--- LLM-as-judge ---
A: 6/10 — More complete across all three sections with better detail on key people and product launches, but includes speculative/hallucinated future rounds and unverified figures like Andrej Karpathy as an Anthropic researcher.

B: 5/10 — Includes the important Amazon investment detail missing from A and correctly names John Schulman, but the product launches section is significantly thinner and also contains speculative future funding rounds presented as fact.

Verdict: A wins — Despite shared weaknesses around speculative late-stage funding data, Answer A provides a more complete and memo-ready briefing across all three required sections, particularly in key people and product launches coverage.


## How to think about it

Three rules cover every tool you'll write:

1. **Return a string.** The middleware only compresses string outputs. JSON-encode dicts before returning if you want them compressed.
2. **Write a good docstring.** It becomes the tool's description — the LLM uses it to decide when to call your tool.
3. **Don't compress yourself.** Compression is automatic at the middleware layer. Calling `client.compress(...)` inside a tool will double-compress.

## Tuning compression

| Key | Default | Effect |
|---|---|---|
| `target_compression_ratio` | 0.5 | 0–1 fraction to remove (0.7 = aggressive). `>1` means Nx. |
| `min_tokens` | 200 | Skip compression for outputs shorter than this. |
| `coarse` | server default `True` | Paragraph-level (faster) vs token-level. |
| `allow_tools` | `None` | Whitelist of tool names to compress. |
| `ignore_tools` | `None` | Blacklist of tool names. |
| `on_error` | `"passthrough"` | `"raise"` to surface backend errors instead of returning original. |


## Per-call LLM knobs

Every facade forwards the common chat-model knobs to the underlying provider on a per-call basis — `temperature`, `top_p`, `top_k`, `max_tokens`, `stop_sequences`, `presence_penalty`, `frequency_penalty`, `seed`, `logprobs`, `top_logprobs`. They're applied via LangChain's `chat.bind(...)` so the cached chat model is never mutated and concurrent calls with different knobs don't interfere.

```python
client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=512,
    temperature=0.2,
    top_p=0.9,
    messages=[{"role": "user", "content": "..."}],
)
```

Gemini's `max_output_tokens` is aliased automatically when targeting `llm="google_genai:..."`. Anything else not in the recognized set (e.g. provider-specific extras) flows through `**kw` to the engine call.